# LoRA Fine-Tuning Pipeline - Data Preparation
**Author:** S.S. Tarek  
**Project:** LLM Fine-Tuning with LoRA - Phi-3 Mini on Structured JSON Output  

## Objective

This notebook handles the data preparation pipeline for fine-tuning Phi-3 Mini using LoRA.

The task is **structured JSON output generation** - given a short emotional text, the model 
must output a valid JSON object matching a fixed schema with four fields:
- `emotion` - the primary emotion expressed
- `intensity` - how strongly the emotion is expressed
- `formality` - whether the language is formal or informal
- `actionable` - whether the text implies someone should respond or help

### Why This Task?
The base model (Phi-3 Mini) has no training signal for this specific schema definition. 
It understands emotion but produces inconsistent, often unparseable output when asked to 
follow a strict JSON format. LoRA fine-tuning teaches the model to reliably follow the schema.

### Pipeline Overview
1. Load the `dair-ai/emotion` dataset from HuggingFace as the source of input texts
2. Use AWS Bedrock (Claude Haiku) to enrich each example with schema-aligned labels
3. Validate and filter outputs, save to JSONL with checkpointing
4. Perform stratified train/val/test split
5. Upload splits to S3 for EC2 training

### Data Note
Raw dataset rows are never committed to this repository in compliance with the 
`dair-ai/emotion` dataset license. The generation script is fully 
reproducible - run it to regenerate the dataset.

## Imports and Environment Setup

All AWS credentials and configuration are loaded from a `.env` file (not committed to 
the repository). See `.env.example` for required variables.

In [36]:
import os
import json
import re
from pathlib import Path
import boto3
from dotenv import load_dotenv
from datasets import load_dataset
import pandas as pd
import random
from collections import defaultdict
import time
import jsonlines
from pathlib import Path
from tqdm import tqdm
import boto3
import os

In [ ]:
load_dotenv()
print("Access Key loaded:", "AWS_ACCESS_KEY_ID" in os.environ)
print("Secret Key loaded:", "AWS_SECRET_ACCESS_KEY" in os.environ)
print("Region:", os.getenv("AWS_DEFAULT_REGION"))

## Bedrock Client

We use AWS Bedrock with Claude Haiku via a cross-region inference profile. 
The client is initialized with explicit credentials loaded from the environment.

In [5]:
bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=os.getenv("AWS_DEFAULT_REGION"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY")
)
BEDROCK_MODEL_ID=os.getenv("BEDROCK_MODEL_ID")
print("Bedrock client created successfully")
print("Bedrock Model ID: ",BEDROCK_MODEL_ID )

Bedrock client created successfully
Bedrock Model ID:  apac.anthropic.claude-3-haiku-20240307-v1:0


## Dataset - dair-ai/emotion

We use the `dair-ai/emotion` dataset as our source of input texts. It contains 
short social media style sentences labeled with one of six emotion classes:
`sadness`, `joy`, `love`, `anger`, `fear`, `surprise`.

The existing labels serve as an anchor for Claude's enrichment - ensuring the 
`emotion` field in our schema is grounded in human-verified labels rather than 
purely model-generated ones.

In [6]:
emotion_ds = load_dataset("dair-ai/emotion")

label_names = emotion_ds["train"].features["label"].names

print(emotion_ds)
print("\nLabel names:", label_names)
print("Train rows :", len(emotion_ds["train"]))
print("Val rows   :", len(emotion_ds["validation"]))
print("Test rows  :", len(emotion_ds["test"]))

# Spot check 5 rows
rows = []
for i in range(5):
    row = emotion_ds["train"][i]
    rows.append({
        "text"     : row["text"],
        "label_id" : row["label"],
        "emotion"  : label_names[row["label"]],
    })

pd.DataFrame(rows)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Label names: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
Train rows : 16000
Val rows   : 2000
Test rows  : 2000


,text,label_id,emotion
0,i didnt feel humiliated,0,sadness
1,i can go from feeling so hopeless to so damned...,0,sadness
2,im grabbing a minute to post i feel greedy wrong,3,anger
3,i am ever feeling nostalgic about the fireplac...,2,love
4,i am feeling grouchy,3,anger


## Schema Enrichment via Bedrock

Each dataset example is sent to Claude Haiku with:
- The input text
- The known base emotion label as an anchor

Claude returns a JSON object with all four schema fields. We use a system prompt 
to enforce strict JSON output and extract the result with a regex-based parser 
that handles any markdown wrapping Claude may add.

### Why Claude as the Labeler?
Generating labels this way introduces Claude's labeling bias into the training signal. 
The model learns to imitate Claude's interpretation of `intensity`, `formality`, and 
`actionable` - not a ground truth human annotation. This is a known limitation of this 
synthetic label generation.

In [10]:
def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    return json.loads(match.group())

In [13]:
def call_claude(text: str, base_emotion: str) -> dict:
    system_prompt = """You are a strict data labeling assistant.
Return ONLY valid JSON matching exactly this schema:
{
  "emotion"   : "sadness | joy | love | anger | fear | surprise",
  "intensity" : "low | medium | high",
  "formality" : "formal | informal",
  "actionable": true | false
}
Rules:
- emotion must exactly match the known base emotion provided by the user.
- intensity reflects how emotionally strong the text sounds.
- formality reflects whether the wording is formal or informal.
- actionable is true if the text implies someone should respond or help.
- No extra fields. No markdown. No explanation. No extra text."""

    user_content = f"Text: {text}\nBase emotion: {base_emotion}"

    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 200,
        "temperature": 0,
        "system": system_prompt,
        "messages": [
            {
                "role": "user",
                "content": user_content
            }
        ]
    }, ensure_ascii=False)

    response = bedrock_client.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=body,
        contentType="application/json",
        accept="application/json",
    )

    raw_text = json.loads(response["body"].read())["content"][0]["text"]
    parsed   = extract_json(raw_text)

    return {"raw": raw_text, "parsed": parsed}



## Sanity Check - Single Example

Before running batch generation, we verify the full pipeline on a single example:
- Bedrock client is reachable
- Claude returns parseable JSON
- Schema fields match expected values

In [14]:
row=emotion_ds["train"][0]
text=row["text"]
base_emotion = label_names[row["label"]]

print("Input text  :", text)
print("Base emotion:", base_emotion)

result =call_claude(text, base_emotion)

print("\nRaw response:")
print(result["raw"])
print("\nParsed:")
result["parsed"]

Input text  : i didnt feel humiliated
Base emotion: sadness

Raw response:
{
  "emotion"   : "sadness",
  "intensity" : "medium",
  "formality" : "informal",
  "actionable": false
}

Parsed:


{'emotion': 'sadness',
 'intensity': 'medium',
 'formality': 'informal',
 'actionable': False}

## Batch Generation with Checkpointing

We generate enriched labels for 1,200 examples with:
- **Checkpointing** - progress is saved to JSONL after each example. If the process 
  is interrupted, it resumes from where it left off
- **Validation** - each output is checked against the schema before saving. 
  Invalid outputs are silently skipped
- **Rate limiting** - a 0.3 second delay between API calls to avoid throttling

Of the 1,200 attempted examples, 1,174 passed validation after filtering.

In [47]:
# --- Config ---
N_TOTAL= 1200
DATA_DIR_NAME= "data"
RAW_OUTPUT_NAME= "generated_raw.jsonl"

DATA_DIR= Path(DATA_DIR_NAME)
OUTPUT_FILE= DATA_DIR / RAW_OUTPUT_NAME
DATA_DIR.mkdir(parents=True, exist_ok=True)

VALID_EMOTIONS= {"sadness", "joy", "love", "anger", "fear", "surprise"}
VALID_INTENSITIES= {"low", "medium", "high"}
VALID_FORMALITIES= {"formal", "informal"}

# Resume from checkpoint if file exists
already_done = set()
if OUTPUT_FILE.exists():
    with jsonlines.open(OUTPUT_FILE) as reader:
        for obj in reader:
            already_done.add(obj["idx"])
    print(f"Resuming - {len(already_done)} examples already done")
else:
    print("Starting fresh")


def validate(parsed: dict, base_emotion: str) -> bool:
    """Returns True only if all fields are present and valid."""
    if parsed.get("emotion")    not in VALID_EMOTIONS:
        return False
    if parsed.get("intensity")  not in VALID_INTENSITIES:
        return False
    if parsed.get("formality")  not in VALID_FORMALITIES:
        return False
    if not isinstance(parsed.get("actionable"), bool):
        return False
    # emotion must match base label
    if parsed["emotion"] != base_emotion:
        return False
    return True


# --- Batch generation ---
failed  = 0
success = 0

with jsonlines.open(OUTPUT_FILE, mode="a") as writer:
    for idx in tqdm(range(N_TOTAL)):
        if idx in already_done:
            success += 1
            continue

        row = emotion_ds["train"][idx]
        text= row["text"]
        base_emotion=label_names[row["label"]]

        try:
            result = call_claude(text, base_emotion)
            parsed = result["parsed"]

            if not validate(parsed, base_emotion):
                failed += 1
                continue

            writer.write({
                "idx"      : idx,
                "text"     : text,
                "emotion"  : parsed["emotion"],
                "intensity": parsed["intensity"],
                "formality": parsed["formality"],
                "actionable": parsed["actionable"],
            })
            success += 1

        except Exception as e:
            failed += 1
            continue

        # Small delay to avoid throttling
        time.sleep(0.3)
print(f"\nDone. Success: {success}  |  Failed/skipped: {failed}")

Resuming - 1171 examples already done


100%|███████████████████████████████████████████████████████████████████████████████| 1200/1200 [00:18<00:00, 64.44it/s]


Done. Success: 1174  |  Failed/skipped: 26


## Stratified Train / Val / Test Split

We perform a stratified split by emotion class to ensure proportional representation 
across all three splits. A plain random shuffle would risk underrepresenting rare 
classes like `surprise` (33 total examples) in the test set.

### Split Sizes
| Split | Examples |
|-------|----------|
| Train | 782      |
| Val   | 196      |
| Test  | 196      |


In [23]:
DATA_DIR   = Path("data")
INPUT_FILE = DATA_DIR / "generated_raw.jsonl"
SEED       = 42

# --- Load ---
with jsonlines.open(INPUT_FILE) as reader:
    all_examples = list(reader)

print(f"Total examples loaded: {len(all_examples)}")

# --- Group by emotion ---
by_emotion = defaultdict(list)
for ex in all_examples:
    by_emotion[ex["emotion"]].append(ex)

# --- Stratified split ---
train, val, test = [], [], []

random.seed(SEED)
for emotion, examples in by_emotion.items():
    random.shuffle(examples)
    n     = len(examples)
    n_val  = max(1, int(n * 0.17))   # ~17% val
    n_test = max(1, int(n * 0.17))   # ~17% test
    n_train = n - n_val - n_test

    train.extend(examples[:n_train])
    val.extend(examples[n_train:n_train + n_val])
    test.extend(examples[n_train + n_val:])

# Shuffle each split
random.shuffle(train)
random.shuffle(val)
random.shuffle(test)

print(f"Train : {len(train)}")
print(f"Val   : {len(val)}")
print(f"Test  : {len(test)}")

# Class distribution check
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    counts = defaultdict(int)
    for ex in split_data:
        counts[ex["emotion"]] += 1
    print(f"\n{split_name} distribution:")
    for emotion, count in sorted(counts.items()):
        print(f"  {emotion:10s}: {count}")

# Save
for split_name, split_data in [("train", train), ("val", val), ("test", test)]:
    out_path = DATA_DIR / f"{split_name}.jsonl"
    with jsonlines.open(out_path, mode="w") as writer:
        writer.write_all(split_data)
    print(f"\nSaved {split_name}.jsonl → {out_path}")

Total examples loaded: 1174
Train : 782
Val   : 196
Test  : 196

train distribution:
  anger     : 109
  fear      : 91
  joy       : 271
  love      : 74
  sadness   : 204
  surprise  : 33

val distribution:
  anger     : 27
  fear      : 23
  joy       : 69
  love      : 18
  sadness   : 52
  surprise  : 7

test distribution:
  anger     : 27
  fear      : 23
  joy       : 69
  love      : 18
  sadness   : 52
  surprise  : 7

Saved train.jsonl → data/train.jsonl

Saved val.jsonl → data/val.jsonl

Saved test.jsonl → data/test.jsonl


**Note:** `surprise` is severely underrepresented (33 train, 7 test examples). Per-class 
metrics for `surprise` should be interpreted with caution.

## Upload to S3

Splits are uploaded to S3 for access during EC2 training. The EC2 instance 
downloads only what it needs at runtime - no data is hardcoded into the 
training environment.

In [27]:
S3_BUCKET = os.getenv("S3_BUCKET")
S3_PREFIX = "lora-project"

s3_client = boto3.client(
    "s3",
    region_name=os.getenv("AWS_DEFAULT_REGION"),
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY_ID"),
    aws_secret_access_key=os.getenv("AWS_SECRET_ACCESS_KEY"),
)

splits = ["train", "val", "test"]

for split in splits:
    local_path = Path("data") / f"{split}.jsonl"
    s3_key     = f"{S3_PREFIX}/data/{split}.jsonl"

    s3_client.upload_file(
        Filename=str(local_path),
        Bucket=S3_BUCKET,
        Key=s3_key,
    )
    print(f"Uploaded {local_path} → s3://{S3_BUCKET}/{s3_key}")

Uploaded data/train.jsonl → s3://caws-poc-alert-refinement/lora-project/data/train.jsonl
Uploaded data/val.jsonl → s3://caws-poc-alert-refinement/lora-project/data/val.jsonl
Uploaded data/test.jsonl → s3://caws-poc-alert-refinement/lora-project/data/test.jsonl


## Citations

**Dataset**
```bibtex
@inproceedings{saravia-etal-2018-carer,
    title = "{CARER}: Contextualized Affect Representations for Emotion Recognition",
    author = "Saravia, Elvis  and
      Liu, Hsien-Chi Toby  and
      Huang, Yen-Hao  and
      Wu, Junlin  and
      Chen, Yi-Shin",
    booktitle = "Proceedings of the 2018 Conference on Empirical Methods in Natural Language Processing",
    month = oct # "-" # nov,
    year = "2018",
    address = "Brussels, Belgium",
    publisher = "Association for Computational Linguistics",
    url = "https://www.aclweb.org/anthology/D18-1404",
    doi = "10.18653/v1/D18-1404",
    pages = "3687--3697"
}
```
**Base Model** : Microsoft Phi-3 Mini 4k Instruct - https://huggingface.co/microsoft/Phi-3-mini-4k-instruct 

**Labeling Model** : Anthropic Claude Haiku via AWS Bedrock